In [ ]:
# =============================================================
# Adversarial Attacks Series — Note 06
# Architectural Awareness: Building the Sensor into the Boat
# =============================================================
#
# Series:  Humble Model / Architectural Awareness
# Dataset: CIFAR-10
# Model:   CNN (same as Essay #5)
#
# Notebook structure:
#   Part A: Imports and Setup
#   Part B: Dataset Loading (CIFAR-10)
#   Part C: Model Architecture (shared backbone)
#   Part D: Baseline Model (load or train)
#   Part E: Multiple Prediction Heads
#   Part F: Evidential Outputs
#   Part G: Boundary Distance (re-run from Essay #5, with held-out split)
#   Part H: Head-to-Head Comparison
#   Part I: Fused Gate Evaluation
#   Part J: Results and Summary
# =============================================================

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part A: Imports and Setup
# ─────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm
import random
import pandas as pd

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part B: Dataset Loading (CIFAR-10) — with held-out split
# ─────────────────────────────────────────────────────────────

cifar_transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

cifar_transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform_test)

# Held-out split: 20% calibration, 80% evaluation
calib_size = int(0.2 * len(test_dataset))
eval_size = len(test_dataset) - calib_size
calib_dataset, eval_dataset = random_split(test_dataset, [calib_size, eval_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
calib_loader = DataLoader(calib_dataset, batch_size=64, shuffle=False, num_workers=0)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset):,}")
print(f"Calibration samples: {len(calib_dataset):,}")
print(f"Evaluation samples: {len(eval_dataset):,}")

# Per-channel valid normalized range
CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2023, 0.1994, 0.2010]).view(1, 3, 1, 1)
CIFAR_MIN = ((0 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)
CIFAR_MAX = ((1 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)

def clamp_valid(x):
    return torch.max(torch.min(x, CIFAR_MAX), CIFAR_MIN)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part C: Model Architecture (Shared Backbone)
# ─────────────────────────────────────────────────────────────

class BackboneCNN(nn.Module):
    """Shared backbone for all directions."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv3(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return x

class StandardCNN(nn.Module):
    """Standard CNN for baseline (same as Essay #5)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = BackboneCNN()
        self.fc_out = nn.Linear(256, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        return self.fc_out(features)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part D: Baseline Model (load or train)
# ─────────────────────────────────────────────────────────────

def train_model(model, train_loader, epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        avg_loss = running_loss / len(train_loader)
        print(f"📊 Epoch {epoch} — avg loss: {avg_loss:.4f}")
    return model

def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

baseline_model = StandardCNN().to(DEVICE)

if os.path.exists('checkpoint_baseline_cifar10.pth'):
    checkpoint = torch.load('checkpoint_baseline_cifar10.pth', map_location=DEVICE)
    baseline_model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Loaded baseline model from checkpoint")
else:
    print("🆕 Training baseline model from scratch...")
    baseline_model = train_model(baseline_model, train_loader, epochs=20)
    torch.save({'model_state_dict': baseline_model.state_dict()}, 'checkpoint_baseline_cifar10.pth')
    print("💾 Saved baseline model checkpoint")

clean_acc = evaluate_accuracy(baseline_model, eval_loader)
print(f"📈 Baseline clean test accuracy: {clean_acc:.2f}%")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part E: Multiple Prediction Heads
# ─────────────────────────────────────────────────────────────

class MultiHeadCNN(nn.Module):
    """CNN with shared backbone and multiple independent heads."""
    def __init__(self, backbone, num_heads=5, num_classes=10):
        super().__init__()
        self.backbone = backbone
        self.heads = nn.ModuleList([
            nn.Linear(256, num_classes) for _ in range(num_heads)
        ])

    def forward(self, x):
        features = self.backbone(x)
        logits = torch.stack([head(features) for head in self.heads], dim=1)
        return logits  # [batch, num_heads, num_classes]

    def forward_with_disagreement(self, x):
        """Return mean prediction and variance across heads."""
        logits = self.forward(x)
        probs = F.softmax(logits, dim=2)  # [batch, num_heads, num_classes]
        mean_probs = probs.mean(dim=1)    # [batch, num_classes]
        var_probs = probs.var(dim=1)      # [batch, num_classes]
        return mean_probs, var_probs

# Train multi-head model
multi_head_model = MultiHeadCNN(BackboneCNN(), num_heads=5).to(DEVICE)

if os.path.exists('checkpoint_multihead.pth'):
    checkpoint = torch.load('checkpoint_multihead.pth', map_location=DEVICE)
    multi_head_model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Loaded multi-head model from checkpoint")
else:
    print("🆕 Training multi-head model...")
    # Train the backbone and heads jointly
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(multi_head_model.parameters(), lr=0.001)
    multi_head_model.train()
    for epoch in range(20):
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = multi_head_model(images)  # [batch, num_heads, num_classes]
            # Average loss across heads
            loss = torch.stack([criterion(logits[:, h, :], labels) for h in range(5)]).mean()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        avg_loss = running_loss / len(train_loader)
        print(f"📊 Epoch {epoch} — avg loss: {avg_loss:.4f}")
    torch.save({'model_state_dict': multi_head_model.state_dict()}, 'checkpoint_multihead.pth')
    print("💾 Saved multi-head model checkpoint")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part F: Evidential Outputs
# ─────────────────────────────────────────────────────────────

class EvidentialCNN(nn.Module):
    """CNN with Dirichlet evidential output."""
    def __init__(self, backbone, num_classes=10):
        super().__init__()
        self.backbone = backbone
        self.fc_alpha = nn.Linear(256, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        alpha = F.softplus(self.fc_alpha(features)) + 1.0  # ensure positive
        return alpha  # [batch, num_classes]

    def forward_with_evidence(self, x):
        alpha = self.forward(x)
        S = alpha.sum(dim=1, keepdim=True)  # total evidence
        probs = alpha / S  # mean of Dirichlet
        evidence = S.squeeze(1)  # total evidence per sample
        return probs, evidence

# Train evidential model
evidential_model = EvidentialCNN(BackboneCNN()).to(DEVICE)

def evidential_loss(alpha, labels, num_classes=10):
    """Loss for evidential deep learning (Sensoy et al., 2018)."""
    S = alpha.sum(dim=1, keepdim=True)
    # Cross-entropy-like term
    loss = (labels * (torch.digamma(S) - torch.digamma(alpha))).sum(dim=1)
    # KL divergence term (simplified)
    return loss.mean()

if os.path.exists('checkpoint_evidential.pth'):
    checkpoint = torch.load('checkpoint_evidential.pth', map_location=DEVICE)
    evidential_model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Loaded evidential model from checkpoint")
else:
    print("🆕 Training evidential model...")
    optimizer = optim.Adam(evidential_model.parameters(), lr=0.001)
    evidential_model.train()
    for epoch in range(20):
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            alpha = evidential_model(images)
            loss = evidential_loss(alpha, F.one_hot(labels, num_classes=10).float())
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        avg_loss = running_loss / len(train_loader)
        print(f"📊 Epoch {epoch} — avg loss: {avg_loss:.4f}")
    torch.save({'model_state_dict': evidential_model.state_dict()}, 'checkpoint_evidential.pth')
    print("💾 Saved evidential model checkpoint")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Part G: Boundary Distance (re-run with held-out split)
# ─────────────────────────────────────────────────────────────

def fgsm_attack(model, images, labels, epsilon=0.03):
    model.eval()
    images = images.clone().detach().to(DEVICE)
    images.requires_grad = True
    outputs = model(images)
    loss = nn.CrossEntropyLoss()(outputs, labels.to(DEVICE))
    model.zero_grad()
    loss.backward()
    perturbed = images + epsilon * images.grad.sign()
    return clamp_valid(perturbed).detach()

def estimate_boundary_distance(model, image, label, max_iters=30):
    model.eval()
    image = image.clone().detach().to(DEVICE)
    image.requires_grad = True
    output = model(image)
    loss = nn.CrossEntropyLoss()(output, torch.tensor([label], device=DEVICE))
    model.zero_grad()
    loss.backward()
    grad = image.grad.data.sign()
    low, high = 0.0, 1.0
    for _ in range(max_iters):
        mid = (low + high) / 2
        perturbed = image + mid * grad
        perturbed = clamp_valid(perturbed)
        with torch.no_grad():
            pred = model(perturbed).argmax().item()
        if pred != label:
            high = mid
        else:
            low = mid
    return (low + high) / 2


In [ ]:
# ─────────────────────────────────────────────────────────────
# Part H: Head-to-Head Comparison
# ─────────────────────────────────────────────────────────────

'''We will collect:
 - Boundary distance scores
 - Multi-head disagreement (variance across heads)
 - Evidential evidence (total evidence per sample)
 - Confidence (for baseline)'''

# ... (collection and evaluation functions)
# To be implemented based on your results
print("Head-to-Head comparison will be implemented after running all methods.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part I: Fused Gate Evaluation
# ─────────────────────────────────────────────────────────────

''' We will evaluate fused gates combining confidence, boundary distance,
 and architectural signals.'''

print("Fused gate evaluation will be implemented after running all methods.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part J: Results and Summary
# ─────────────────────────────────────────────────────────────

print("\n" + "="*55)
print("📋 Experiment Summary")
print("="*55)

# Placeholder for results
print("""
Multi-Head CNN:
  - Clean accuracy: [TBD]
  - Adversarial accuracy: [TBD]
  - Clean deferral: [TBD]
  - Adversarial deferral: [TBD]
  - Adversarial risk: [TBD]

Evidential CNN:
  - Clean accuracy: [TBD]
  - Adversarial accuracy: [TBD]
  - Clean deferral: [TBD]
  - Adversarial deferral: [TBD]
  - Adversarial risk: [TBD]

Boundary Distance:
  - Clean deferral: [TBD]
  - Adversarial deferral: [TBD]
  - Adversarial risk: [TBD]
""")

print("\n✅ Notebook complete!")